<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch13_ex9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tweak the `Seq2SeqModel` model to forecas both rail and train ridership for the next 14 days.

## Download the data

In [30]:
import pandas as pd
from pathlib import Path
import tarfile
import urllib.request

def download_and_extract_ridership_data():
    tarball_path = Path("datasets/ridership.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/ridership.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets", filter="data")

download_and_extract_ridership_data()

In [31]:
path =  Path("datasets/ridership/CTA_-_Ridership_-_Daily_Boarding_Totals.csv")
df = pd.read_csv(path, parse_dates=["service_date"])


df.head()

,service_date,day_type,bus,rail_boardings,total_rides
0,2001-01-01,U,297192,126455,423647
1,2001-01-02,W,780827,501952,1282779
2,2001-01-03,W,824923,536432,1361355
3,2001-01-04,W,870021,550011,1420032
4,2001-01-05,W,890426,557917,1448343


In [32]:
# rename the columns
df.columns = ["date", "day_type", "bus", "rail", "total"]
df.head()

,date,day_type,bus,rail,total
0,2001-01-01,U,297192,126455,423647
1,2001-01-02,W,780827,501952,1282779
2,2001-01-03,W,824923,536432,1361355
3,2001-01-04,W,870021,550011,1420032
4,2001-01-05,W,890426,557917,1448343


In [33]:
df = df.drop("total", axis=1) # no need fot the total = bus+train

In [34]:
df = df.drop_duplicates()

## TimeSeriesDataset

In [35]:
import torch

# this class chops a given time series into all possible windows of
# a given length, each with the day after as target
class TimeSeriesDataset(torch.utils.data.Dataset):
  def __init__(self, series, window_length):
    self.series = series #e.g 1000 days rail ridership
    self.window_length=window_length #e.g 56 days

  def __len__(self):
    # how many training examples can we extract
    return len(self.series) - self.window_length

  def __getitem__(self, index):
    if index >= len(self): # example no. index (e.g 2)
      raise IndexError("dataset index out of range")

    end = index + self.window_length #1st index after window (e.g. 58)
    window = self.series[index:end] # e.g. elements [2, 3, 4, ..., 57]
    target = self.series[end] # e.g element 58, the target
    return window, target

## Evaluate and train function

In [36]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [37]:
%pip install torchmetrics

In [38]:
import torchmetrics
import torch.nn as nn

In [39]:
# from ch12_ex8
def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch) #update at each iteration
  return metric.compute()

import time
def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, factor=0.1, patience=10):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=factor, patience=patience)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())

        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)

        scheduler.step(val_metric)

        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")

    return history

## Fit and evaluate function

In [40]:
def fit_and_evaluate(model, train_loader, valid_loader, lr, n_epochs=50,
                    factor=0.1):
    loss_fn = nn.HuberLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    metric = torchmetrics.MeanAbsoluteError().to(device)
    history = train(model, optimizer, loss_fn, metric,
                    train_loader, valid_loader, n_epochs=n_epochs,
                    factor=factor)
    return min(history["valid_metrics"]) * 1e6

## ForecastAheadDataset

In [41]:
# Set the date as the index
df = df.sort_values("date").set_index("date")

In [42]:
df_mulvar = df [["rail", "bus"]]/1e6
df_mulvar["next_day_type"] = df["day_type"].shift(-1)
df_mulvar = pd.get_dummies(df_mulvar, dtype=float) # one-hot encode day type

In [43]:
df_mulvar.head()

,rail,bus,next_day_type_A,next_day_type_U,next_day_type_W
date,,,,,
2001-01-01,0.126455,0.297192,0.0,0.0,1.0
2001-01-02,0.501952,0.780827,0.0,0.0,1.0
2001-01-03,0.536432,0.824923,0.0,0.0,1.0
2001-01-04,0.550011,0.870021,0.0,0.0,1.0
2001-01-05,0.557917,0.890426,1.0,0.0,0.0


In [44]:
mulvar_train = torch.FloatTensor(df_mulvar["2016-01":"2018-12"].values)
mulvar_valid = torch.FloatTensor(df_mulvar["2019-01":"2019-05"].values)
mulvar_test = torch.FloatTensor(df_mulvar["2019-06":].values)

In [45]:
#modify the TimeSeriesDataset to output 14 days as the target
# the input will be multivariate
class ForecastAheadDataset(TimeSeriesDataset):
  def __len__(self):
    return len(self.series)-self.window_length-14+1
    # suppose we have 16 days and the training window is 2 days: it should be a valid
    # member of the dataset but 16-2-14 =0, so we need to add a +1

  def __getitem__(self, index):
    end = index + self.window_length
    window = self.series[index:end]
    target = self.series[end:end+14, 0] # 0: rail ridership
    return window, target

## Seq2Seq Dataset

In [46]:
# the target is now a sequence of consecutive windows,
# shifted by one time tep at each time step
class Seq2SeqDataset(ForecastAheadDataset):
  def __getitem__(self, index):
    end = index + self.window_length
    window = self.series[index:end]
    target_period = self.series[index+1:end+14, :2] # cols 0,1: rail, bus
    target = target_period.unfold(dimension=0, size=14, step=1)
    # now target is of shape (window_length, 2, 14), that is,
    # a (2,14) block at each time stamp
    # To match the time series logic of this data, it should be
    # a 2-variate time series of shape (length, dimensionality),
    # that is, (14,2)
    target =  target.permute(0,2,1)
    return window, target

## DataLoaders using Seq2SeqDataset

In [47]:
from torch.utils.data import DataLoader

window_length = 56
batch_size = 32

train_dataset = Seq2SeqDataset(mulvar_train, window_length)
valid_dataset = Seq2SeqDataset(mulvar_valid, window_length)
test_dataset = Seq2SeqDataset(mulvar_test, window_length)

train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size)
test_loader = DataLoader(test_dataset, batch_size)

##SimpleRNNModel: seq2vec forecast

In [48]:
class SimpleRnnModel(nn.Module):
  # input size is dimension of the time series. =1 for univariate
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size # neurons in the rnn layer

    # input size = 5 (5-variate time series)
    self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)

    self.output = nn.Linear(hidden_size, output_size)

  def forward(self, X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:,-1])

In this code output size is 14, that is the length of the prediction. However, when talking about the seq2seq architecture, especially how the training data is constructed, 14 is the dimensionality of the timeseries of shape (54, 14)!

Now, here we want to predict 2 values, and in fact the `Seq2SeqDataset` spits out time series of shape (54,14,2), so at each time point the target is of shape (14,2).

Therefore, the output dimension of the `nn.Linear` layer is just 28. That is, the nn.Linear layer has to predict 28 values. The idea is to reshape the outpur of the linear layer as follows:

`target`:	`(batch, window, 14,2)`

`Linear output (raw)`:	`(batch, window_length, 28)`

`Linear output (reshaped)`:	`(batch, window_length, 14, 2)`

## Seq2SeqModel: seq2seq forecast

In [51]:
# we modify the SimpleRnnModel so that the output nn.Linear
# is applied to the output at each time step

# as discussed above, the nn.Linear layer just needs to predict 14*2 values.

class Seq2SeqModel(SimpleRnnModel):
    def __init__(self, input_size, hidden_size, forecast_horizon, num_vars):
        super().__init__(input_size, hidden_size, output_size=forecast_horizon * num_vars)
        self.horizon = forecast_horizon
        self.num_vars = num_vars

    def forward(self, X):
      outputs, last_state = self.rnn(X)
      raw_output = self.output(outputs) #(batch, window length, 28)
      reshaped_output = raw_output.reshape(*raw_output.shape[:-1], self.horizon, self.num_vars) #(batch, window length, 14, 2)
      return reshaped_output

## Train the model

In [52]:
torch.manual_seed(42)
seq_model = Seq2SeqModel(input_size=5, hidden_size=32, forecast_horizon=14, num_vars=2)
seq_model = seq_model.to(device)

fit_and_evaluate(seq_model, train_loader, valid_loader, 0.01, 75, 0.5)

Epoch 1/75, train loss: 0.0367, train metric: 0.2011, valid metric: 0.1081
Epoch 2/75, train loss: 0.0070, train metric: 0.0882, valid metric: 0.0665
Epoch 3/75, train loss: 0.0045, train metric: 0.0650, valid metric: 0.0614
Epoch 4/75, train loss: 0.0040, train metric: 0.0587, valid metric: 0.0527
Epoch 5/75, train loss: 0.0038, train metric: 0.0557, valid metric: 0.0528
Epoch 6/75, train loss: 0.0037, train metric: 0.0537, valid metric: 0.0617
Epoch 7/75, train loss: 0.0038, train metric: 0.0553, valid metric: 0.0502
Epoch 8/75, train loss: 0.0035, train metric: 0.0526, valid metric: 0.0517
Epoch 9/75, train loss: 0.0036, train metric: 0.0526, valid metric: 0.0534
Epoch 10/75, train loss: 0.0036, train metric: 0.0524, valid metric: 0.0571
Epoch 11/75, train loss: 0.0035, train metric: 0.0524, valid metric: 0.0514
Epoch 12/75, train loss: 0.0035, train metric: 0.0524, valid metric: 0.0604
Epoch 13/75, train loss: 0.0035, train metric: 0.0532, valid metric: 0.0507
Epoch 14/75, train lo

49417.33181476593

In [60]:
seq_model.eval()
with torch.no_grad():
  some_window = mulvar_valid[:window_length] # first window of the validation set
  print("First window shape:", some_window.size())
  X = some_window.unsqueeze(dim=0) # fake batch size 1
  print("First window shape:", X.size())

  Y_preds = seq_model(X.to(device))
  print("Y_preds shape:", Y_preds.size())

  Y_pred = Y_preds[0,-1]
  print("Y_pred shape:", Y_pred.size())


First window shape: torch.Size([56, 5])
First window shape: torch.Size([1, 56, 5])
Y_preds shape: torch.Size([1, 56, 14, 2])
Y_pred shape: torch.Size([14, 2])
